In [0]:
import os
import urllib.request
import jpype
import jpype.imports
import duckdb
import psycopg2
import polars as pl
from datetime import date

# =========================================================
# CONFIGURATION  — all secrets from environment variables
# =========================================================
ORA_WUSER     = os.environ["ORA_WUSER"]
ORA_WPASSWORD = os.environ["ORA_WPASSWORD"]
ORA_WHOST     = os.environ["ORA_WHOST"]
ORA_WSERVICE  = os.environ["ORA_WSERVICE"]

ORA_AUSER     = os.environ["ORA_AUSER"]
ORA_APASSWORD = os.environ["ORA_APASSWORD"]
ORA_AHOST     = os.environ["ORA_AHOST"]
ORA_ASERVICE  = os.environ["ORA_ASERVICE"]

ORA_PORT = int(os.environ.get("ORA_PORT", 1521))

pg_params = {
    "host":     os.environ["PG_HOST"],
    "port":     int(os.environ.get("PG_PORT", 5432)),
    "dbname":   os.environ["PG_DBNAME"],
    "user":     os.environ["PG_USER"],
    "password": os.environ["PG_PASSWORD"],
}
PG_CONN = " ".join(f"{k}={v}" for k, v in pg_params.items())

# =========================================================
# SETUP JDBC DRIVER
# =========================================================
JDBC_DIR = "/tmp/oracle/jdbc"
JAR_PATH = os.path.join(JDBC_DIR, "ojdbc8.jar")
os.makedirs(JDBC_DIR, exist_ok=True)

if not os.path.exists(JAR_PATH):
    print("Downloading ojdbc8.jar...")
    urllib.request.urlretrieve(
        "https://download.oracle.com/otn-pub/otn_software/jdbc/1916/ojdbc8.jar",
        JAR_PATH,
    )
print(f"ojdbc8.jar ready ({os.path.getsize(JAR_PATH):,} bytes)")

# =========================================================
# START JVM (only once per session)
# =========================================================
if not jpype.isJVMStarted():
    jpype.startJVM(classpath=[JAR_PATH])
    print("JVM started")
else:
    print("JVM already running")

print("Configuration complete")

In [ ]:
import java.sql.Types as JT
from java.sql import DriverManager

QUERY = """
    SELECT *
    FROM ORACLE_TABLE_NAME
"""

# ---------------------------------------------------------
# Helpers: type-aware JDBC value reader + DuckDB type mapper
# ---------------------------------------------------------
def jdbc_value(rs, col_index, jdbc_type):
    """
    Return a Python-native value from a JDBC ResultSet column.
    Preserves numeric and timestamp types instead of str()-casting everything.
    Oracle DATE always carries a time component, so we read it as TIMESTAMP.
    """
    obj = rs.getObject(col_index)
    if obj is None:
        return None
    if jdbc_type in (JT.NUMERIC, JT.DECIMAL, JT.BIGINT, JT.INTEGER, JT.SMALLINT):
        return int(rs.getLong(col_index))
    if jdbc_type in (JT.FLOAT, JT.DOUBLE, JT.REAL):
        return float(rs.getDouble(col_index))
    if jdbc_type in (JT.DATE, JT.TIMESTAMP):
        # Oracle DATE includes time — read as Timestamp to preserve it
        ts = rs.getTimestamp(col_index)
        return str(ts) if ts is not None else None   # "2025-05-19 08:30:00.0"
    return str(obj)   # VARCHAR2, CHAR, CLOB fallback


def jdbc_type_to_duckdb(jdbc_type):
    """
    Map a JDBC type code to the appropriate DuckDB column type.
    Dates become TIMESTAMP (not DATE) to preserve Oracle's time component.
    """
    if jdbc_type in (JT.NUMERIC, JT.DECIMAL, JT.BIGINT, JT.INTEGER, JT.SMALLINT):
        return "BIGINT"
    if jdbc_type in (JT.FLOAT, JT.DOUBLE, JT.REAL):
        return "DOUBLE"
    if jdbc_type in (JT.DATE, JT.TIMESTAMP):
        return "TIMESTAMP"
    return "VARCHAR"


# ---------------------------------------------------------
# Reusable extractor
# ---------------------------------------------------------
def extract_from_oracle(host, port, service, user, password, label):
    """Extract rows from Oracle via JDBC, preserving native column types."""
    jdbc_url = f"jdbc:oracle:thin:@//{host}:{port}/{service}"
    conn = DriverManager.getConnection(jdbc_url, user, password)
    try:
        stmt = conn.createStatement()
        rs   = stmt.executeQuery(QUERY)
        meta = rs.getMetaData()
        n    = meta.getColumnCount()

        columns    = [str(meta.getColumnName(i + 1)).lower() for i in range(n)]
        jdbc_types = [meta.getColumnType(i + 1) for i in range(n)]

        rows = []
        while rs.next():
            rows.append(
                tuple(jdbc_value(rs, i + 1, jdbc_types[i]) for i in range(n))
            )
        rs.close()
        stmt.close()
    finally:
        conn.close()

    print(f"{label} — {len(rows):,} rows, {len(columns)} columns")
    return columns, jdbc_types, rows


columns1, types1, rows1 = extract_from_oracle(
    ORA_WHOST, ORA_PORT, ORA_WSERVICE, ORA_WUSER, ORA_WPASSWORD, "Server Oracle1"
)
columns2, types2, rows2 = extract_from_oracle(
    ORA_AHOST, ORA_PORT, ORA_ASERVICE, ORA_AUSER, ORA_APASSWORD, "Server Oracle2"
)

# Guard: abort early if both sources are empty
if not rows1 and not rows2:
    raise SystemExit("Both Oracle servers returned 0 rows — nothing to load.")

In [ ]:
duck = duckdb.connect(database=":memory:")

# ---------------------------------------------------------
# Load helper — uses per-table type metadata, not a shared
# placeholder string, so schemas can legitimately differ.
# ---------------------------------------------------------
def load_to_duckdb(duck, table_name, columns, jdbc_types, rows, plant_tag):
    col_defs     = ", ".join(
        [f'"{c}" {jdbc_type_to_duckdb(t)}' for c, t in zip(columns, jdbc_types)]
    )
    n_cols       = len(columns)
    placeholders = ", ".join(["?"] * (n_cols + 1))   # +1 for 'plants'

    duck.execute(f'CREATE TABLE {table_name} ({col_defs}, "plants" VARCHAR)')

    if rows:
        tagged = [r + (plant_tag,) for r in rows]
        duck.executemany(f'INSERT INTO {table_name} VALUES ({placeholders})', tagged)

    print(f"{len(rows):,} rows loaded into DuckDB [{table_name}]")


load_to_duckdb(duck, "tableName1", columns1, types1, rows1, "o1")
load_to_duckdb(duck, "tableName2", columns2, types2, rows2, "o2")

duck.execute("""
    CREATE TABLE tableName_raw AS
    SELECT * FROM tableName1
    UNION ALL
    SELECT * FROM tableName2
""")

total_raw = duck.execute("SELECT COUNT(*) FROM tableName_raw").fetchone()[0]
print(f"Combined raw table: {total_raw:,} rows from both Oracle servers")

In [ ]:
df = pl.from_arrow(
    duck.execute("SELECT * FROM tableName_raw").fetch_arrow_table()
)
print("Polars DataFrame shape:", df.shape)
print(df.head(3))

In [ ]:
before = df.shape[0]
df = df.unique(subset=["col1"], keep="first")         # "col1" is the example primary key 
after  = df.shape[0]
dropped = before - after

if dropped:
    print(f"Dropped {dropped:,} cross-server duplicate rows.")
print(f"After deduplication: {df.shape[0]:,} rows")

In [ ]:
df = df.with_columns([
    pl.lit(date.today()).alias("etl_date"),             #etl_date is the example new column to add at the end of the pipeline that is the extraction date
])
print("ETL date column added — shape:", df.shape)

In [ ]:
# ---------------------------------------------------------
# Stage table
# ---------------------------------------------------------
# We pass columns through without re-casting.
# TRY_CAST is only used for input/output (BIGINT → INTEGER downcast).
# create_date and update_date remain TIMESTAMP — not DATE — so the
# time component from Oracle is preserved and not silently truncated.
# ---------------------------------------------------------
duck.register("pl_df", df)

duck.execute("""
    CREATE OR REPLACE TABLE tableName_stage AS
    SELECT
        col1                                 AS col1,
        col2                                 AS col2,
        TRY_CAST(col3      AS INTEGER)       AS col3,
        TRY_CAST(col4      AS INTEGER)       AS col4,
        timestamp                            AS timestamp,
        TRY_CAST(etl_date  AS DATE)          AS etl_date,
        plants                                AS plants
    FROM pl_df
""")

# Audit: flag rows where critical columns came out NULL
bad = duck.execute("""
    SELECT COUNT(*) FROM tableName_stage
    WHERE col1 IS NULL OR col2 IS NULL
""").fetchone()[0]

if bad:
    print(f"WARNING: {bad:,} rows have NULL in critical columns (col1 / col2).")
else:
    print("Stage table created — no critical cast failures.")

print(duck.execute("DESCRIBE tableName_stage").fetchdf().to_string())

In [ ]:
print("Connecting to PostgreSQL...")

duck.execute("INSTALL postgres;")
duck.execute("LOAD postgres;")

try:
    duck.execute("DETACH DATABASE pg;")
except duckdb.CatalogException:
    pass   # not yet attached — that is fine
except Exception as e:
    print(f"Unexpected error detaching pg: {e}")

duck.execute(f"ATTACH '{PG_CONN}' AS pg (TYPE postgres);")

# create_date and update_date are TIMESTAMP (not DATE) because Oracle DATE
# carries a time component. Storing them as DATE in Postgres would silently
# truncate that time on every upsert.
duck.execute("""
    CREATE TABLE IF NOT EXISTS pg.public.tableName_stage (
        col1             BIGINT,
        col2             BIGINT,
        col3             VARCHAR(50),
        col4             INTEGER,
        timestamp        TIMESTAMP,
        etl_date         DATE,
        plants           VARCHAR
    )
""")
print("PostgreSQL connected and target table ensured.")

In [ ]:
%python
print("Running MERGE into PostgreSQL...")

result = duck.execute("""
    MERGE INTO pg.public.tableName AS t
    USING tableName_stage AS s
    ON t.col1 = s.col1

    WHEN MATCHED THEN
        UPDATE SET
            col2       = s.col2,
            col3         = s.col3,
            col4         = s.col4,
            timestamp    = s.timestamp,
            etl_date       = s.etl_date,
            plants         = s.plants

    WHEN NOT MATCHED THEN
        INSERT (
            col1, col2, col3,
            col4, timestamp, etl_date, plants
        )
        VALUES (
            s.col1, s.col2, s.col3,
            s.col4, s.timestamp, s.etl_date, s.plants
        )
""").fetchdf()

print(f"Merge completed.\n{result}")

In [ ]:
with psycopg2.connect(**pg_params) as pg_conn:
    pg_conn.autocommit = True
    with pg_conn.cursor() as cur:
        cur.execute("""
            CREATE UNIQUE INDEX IF NOT EXISTS tableName_pk
            ON public.tableName (col1)
        """)

print("Unique index ensured.")
print("Pipeline completed successfully!")